<a href="https://colab.research.google.com/github/bushrahaider04/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bushrahaider04/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one client_hash_id + one content_hash_id + one report_date.
This represents the daily search performance of a specific content page for a specific client.

Time window: March 1-31, 2026 (month = '2026-03')

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

FEATURES (what I'll use to predict):
• impressions - number of times shown in search results
• clicks - number of times clicked
• position - average search ranking position
• ctr - calculated as clicks / impressions (will create this)

LABEL (what I'm predicting):
• high_performance - 1 if clicks > 10, else 0 (binary classification)

CONTEXT (metadata to understand the row):
• report_date - the date of the performance data
• client_hash_id - pseudonymized client identifier
• content_hash_id - pseudonymized content identifier
• month - partitioning field (always '2026-03' for this window)

EXCLUDED (what I'm ignoring and why):
• client_hash_id - excluded because it's an identifier, not a predictive feature
• content_hash_id - excluded because it's an identifier, not a predictive feature
• month - excluded because I'm only using March 2026, so it has no variance

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
# Install and setup
!pip install duckdb -q

import duckdb
from getpass import getpass

# Connect to DuckDB
con = duckdb.connect()

# This will ask you to paste your token - it won't show on screen
HF_TOKEN = getpass("Paste your Hugging Face token here: ")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Set the path to your data
rel = "hf://datasets/FlyRank/internship-warehouse"

print("=" * 60)
print("ML-04 SEARCH INTELLIGENCE DATA CONTRACT")
print("=" * 60)

# QUERY 1: Grain check
print("\n1. GRAIN CHECK - Should return 0 rows")
print("-" * 40)
try:
    result1 = con.sql(f"""
      SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) as row_count
      FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
      WHERE month = '2026-03'
      GROUP BY report_date, client_hash_id, content_hash_id
      HAVING COUNT(*) > 1
      LIMIT 5
    """)
    print("✅ Grain check passed!")
    print(result1)
except Exception as e:
    print(f"Error: {e}")

# QUERY 2: Row count and date span
print("\n2. ROW COUNT & DATE SPAN")
print("-" * 40)
try:
    result2 = con.sql(f"""
      SELECT
        COUNT(*) as total_rows,
        MIN(report_date) as first_date,
        MAX(report_date) as last_date,
        COUNT(DISTINCT report_date) as unique_days,
        COUNT(DISTINCT client_hash_id) as unique_clients,
        COUNT(DISTINCT content_hash_id) as unique_content
      FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
      WHERE month = '2026-03'
    """)
    print(result2)
except Exception as e:
    print(f"Error: {e}")

# QUERY 3: Availability check
print("\n3. AVAILABILITY CHECK")
print("-" * 40)
try:
    result3 = con.sql(f"""
      SELECT
        COUNT(*) as total_rows,
        COUNTIF(impressions IS NOT NULL AND impressions > 0) as has_impressions,
        COUNTIF(clicks IS NOT NULL AND clicks > 0) as has_clicks,
        COUNTIF(position IS NOT NULL AND position > 0) as has_position,
        COUNTIF(impressions IS NOT NULL AND clicks IS NOT NULL) as has_both
      FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
      WHERE month = '2026-03'
    """)
    print(result3)
except Exception as e:
    print(f"Error: {e}")

# QUERY 4: Window coverage
print("\n4. WINDOW COVERAGE")
print("-" * 40)
try:
    result4 = con.sql(f"""
      WITH all_dates AS (
        SELECT
          report_date,
          COUNT(*) as rows_per_day
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE month = '2026-03'
        GROUP BY report_date
      )
      SELECT
        COUNT(*) as total_days_in_march,
        SUM(CASE WHEN rows_per_day > 0 THEN 1 ELSE 0 END) as days_with_data,
        MIN(report_date) as first_day_with_data,
        MAX(report_date) as last_day_with_data
      FROM all_dates
    """)
    print(result4)
except Exception as e:
    print(f"Error: {e}")

print("\n" + "=" * 60)
print("✅ DATA CONTRACT VERIFIED")
print("=" * 60)

Paste your Hugging Face token here: ··········
ML-04 SEARCH INTELLIGENCE DATA CONTRACT

1. GRAIN CHECK - Should return 0 rows
----------------------------------------
✅ Grain check passed!


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────────┐
│ report_date │ client_hash_id │ content_hash_id │ row_count │
│    date     │    varchar     │     varchar     │   int64   │
├─────────────┴────────────────┴─────────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘


2. ROW COUNT & DATE SPAN
----------------------------------------
┌────────────┬────────────┬────────────┬─────────────┬────────────────┬────────────────┐
│ total_rows │ first_date │ last_date  │ unique_days │ unique_clients │ unique_content │
│   int64    │    date    │    date    │    int64    │     int64      │     int64      │
├────────────┼────────────┼────────────┼─────────────┼────────────────┼────────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │          31 │             55 │         331437 │
└────────────┴────────────┴────────────┴─────────────┴────────────────┴────────────────┘


3. AVAILABILITY CH

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What this data can NEVER tell me:

1. Unbalanced History - I only have March 2026 data, which is just one month. I cannot detect seasonal patterns, week-over-week trends, or long-term performance changes.

2. GSC-only Early Rows - Search Console only retains 16 months of historical data, so I cannot analyze older trends or compare year-over-year performance.

3. Window Overlaps - Using March 2026 data for training means I cannot use the full month to create time-based features that require future knowledge without causing data leakage.

4. No User-Level Behavior - The daily performance table only shows aggregate metrics (total clicks, impressions, position). I cannot track individual user journeys, session durations, or conversion paths.

5. No Query-Level Data - I don't know what specific search terms people used to find this content. This limits my ability to understand search intent.

6. No Content Metadata - I lack page titles, categories, publish dates, author information, or other content attributes that might explain performance differences.

7. No Competitor Benchmarking - I only see performance for this client's content. I cannot compare against competitors or understand market share.

8. Correlation Not Causation - This data can show me patterns (e.g., higher position = more clicks), but it cannot tell me WHY something happened. I can only measure associations, not prove causation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.